In [ ]:
# ---- Optional Colab / Drive setup ----617
# 本地运行时不会触发挂载；在 Colab 中默认启用 Drive，便于恢复 .npz 数据和导出结果 bundle。
import sys
import time
from pathlib import Path

start_time = time.time()
IS_COLAB = "google.colab" in sys.modules

ENABLE_COLAB_DRIVE = True

DRIVE_MOUNT_DIR = Path("/content/drive")
DRIVE_MYDRIVE_DIR = DRIVE_MOUNT_DIR / "MyDrive"
DRIVE_CACHE_ENABLED = bool(IS_COLAB and ENABLE_COLAB_DRIVE)
DRIVE_CACHE_SEARCH_RECURSIVE = True
DRIVE_CACHE_CANDIDATE_DIRS = [
    DRIVE_MYDRIVE_DIR / "EFGP_Eigenpro" / "benchmark_dataset_cache",
    DRIVE_MYDRIVE_DIR / "Colab_Experiments" / "EFGP_Eigenpro" / "benchmark_dataset_cache",
    DRIVE_MYDRIVE_DIR / "benchmark_dataset_cache",
    DRIVE_MYDRIVE_DIR,
]
DRIVE_OUTPUT_DIR = DRIVE_MYDRIVE_DIR / "EFGP_Eigenpro" / "benchmark_exports"

DRIVE_MOUNTED = False
if IS_COLAB and ENABLE_COLAB_DRIVE:
    try:
        from google.colab import drive
        if not DRIVE_MYDRIVE_DIR.exists():
            drive.mount(str(DRIVE_MOUNT_DIR))
        DRIVE_MOUNTED = DRIVE_MYDRIVE_DIR.exists()
    except Exception as e:
        print("Drive mount skipped:", e)
else:
    if IS_COLAB:
        print("[note] Colab detected, but Drive mount is disabled (ENABLE_COLAB_DRIVE=False)")

print("IS_COLAB:", IS_COLAB)
print("ENABLE_COLAB_DRIVE:", ENABLE_COLAB_DRIVE)
print("DRIVE_MOUNTED:", DRIVE_MOUNTED)
print("DRIVE_MYDRIVE_DIR:", DRIVE_MYDRIVE_DIR)
print("DRIVE_CACHE_ENABLED:", DRIVE_CACHE_ENABLED)
print("DRIVE_OUTPUT_DIR:", DRIVE_OUTPUT_DIR)


In [ ]:
## For GitHub import

# ---- Optional GitHub import / package install ----
# 在 Colab 中默认启用：clone/pull 仓库、安装依赖、切到当前 notebook 目录，并打印 GitHub / Colab 链接。
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
ENABLE_COLAB_GITHUB_INSTALL = True

GITHUB_USER = "Yifiwifi"
REPO_NAME = "EFGP-Eigenpro"
GITHUB_BRANCH = "main"
SUB_DIR = "efgp_eigenpro_py"
NOTEBOOK_REL_PATH = "efgp_eigenpro_py/gpu/box_toeplitz_active_block/boxeig_inverse_diagnostics_experiment.ipynb"
NOTEBOOK_DIR_REL = "efgp_eigenpro_py/gpu/box_toeplitz_active_block"
PROJECT_PATH = Path(f"/content/{REPO_NAME}")
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
GITHUB_NOTEBOOK_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}/blob/{GITHUB_BRANCH}/{NOTEBOOK_REL_PATH}"
COLAB_NOTEBOOK_URL = (
    f"https://colab.research.google.com/github/{GITHUB_USER}/{REPO_NAME}"
    f"/blob/{GITHUB_BRANCH}/{NOTEBOOK_REL_PATH}"
)


def _run_cmd(args: list[str], *, cwd: Path | None = None) -> None:
    print("+", " ".join(str(a) for a in args))
    subprocess.run(args, check=True, cwd=None if cwd is None else str(cwd))


if IS_COLAB and ENABLE_COLAB_GITHUB_INSTALL:
    if not PROJECT_PATH.exists():
        _run_cmd(["git", "clone", REPO_URL, str(PROJECT_PATH)])
    else:
        _run_cmd(["git", "pull", "origin", GITHUB_BRANCH], cwd=PROJECT_PATH)

    os.chdir(PROJECT_PATH)

    CODE_ROOT = PROJECT_PATH / SUB_DIR
    REPO_ROOT = PROJECT_PATH
    NOTEBOOK_DIR_PATH = PROJECT_PATH / NOTEBOOK_DIR_REL

    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    if str(CODE_ROOT) not in sys.path:
        sys.path.insert(0, str(CODE_ROOT))

    print("Installing runtime dependencies")
    _run_cmd([
        sys.executable,
        "-m",
        "pip",
        "install",
        "cufinufft",
        "cupy-cuda12x",
        "--extra-index-url",
        "https://pypi.nvidia.com",
    ])
    _run_cmd([sys.executable, "-m", "pip", "install", "git+https://github.com/EigenPro/EigenPro3.git"])
    _run_cmd([sys.executable, "-m", "pip", "install", "git+https://github.com/EigenPro/EigenPro-pytorch.git"])

    requirements_path = CODE_ROOT / "requirements.txt"
    if requirements_path.exists():
        _run_cmd([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])

    if NOTEBOOK_DIR_PATH.exists():
        os.chdir(NOTEBOOK_DIR_PATH)
        print("cwd:", os.getcwd())
    else:
        print("notebook dir not found:", NOTEBOOK_DIR_PATH)

    os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
    try:
        _run_cmd(["ldconfig", "/usr/local/lib"])
    except Exception as e:
        print("ldconfig skipped:", e)

    print("=" * 40)
    try:
        import torch
        import cupy as cp
        import cufinufft
        import eigenpro2
        import eigenpro3

        cp.cuda.Stream.null.synchronize()
        print("PyTorch:", torch.__version__)
        if torch.cuda.is_available():
            print("GPU:", torch.cuda.get_device_name(0))
        else:
            print("GPU: cuda unavailable")
        print("cufinufft / eigenpro2 / eigenpro3 import ok")
    except Exception as e:
        print("runtime check failed:", e)
    print("=" * 40)
else:
    print(
        "[note] GitHub/Colab install cell skipped (IS_COLAB=", IS_COLAB,
        ", ENABLE_COLAB_GITHUB_INSTALL=", ENABLE_COLAB_GITHUB_INSTALL, ")"
    )

print("GitHub notebook URL:", GITHUB_NOTEBOOK_URL)
print("Open in Colab URL:", COLAB_NOTEBOOK_URL)


# BTAB Inverse vs Box-EigenPro Experiments

这个 notebook 把现有 active-block inverse route 和新增的 `btab_boxeig_*` route 放在同一个实验脚本里，便于运行、加载结果、查看诊断指标并导出整理后的表格。

统一约定：`B = expanded low-frequency box = active.box_idx`，`R = B^c = active.tail_idx`。raw active set 只用于构造 box；实际预条件器和 diagnostics 都基于 `B/R`。代码里的 `reg_lambda` 就是线性系统中的 `sigma2`，不会额外乘 `N`。

诊断模式：`none` 不跑 post diagnostics；`cheap` 记录便宜指标；`full` 额外计算 `epsilon_T`、`eta_inv`、`eta_eig`，会更贵。

In [ ]:
from __future__ import annotations

from dataclasses import asdict, replace
from pathlib import Path
from datetime import datetime
import contextlib
import importlib
import json
import re
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)
display(HTML('''
<style>
.output_scroll { height: auto !important; max-height: none !important; }
.jp-OutputArea-output { max-height: none !important; }
.dataframe { width: max-content; min-width: 100%; }
</style>
'''))

NOTEBOOK_ROOT = Path.cwd().resolve()
SEARCH_ROOTS = [
    NOTEBOOK_ROOT,
    *NOTEBOOK_ROOT.parents,
    Path('/content/EFGP-Eigenpro'),
    Path('/content'),
    Path('D:/NU/ML'),
]
REPO_ROOT = None
for root in SEARCH_ROOTS:
    cur = root
    for parent in [cur, *cur.parents]:
        if (parent / 'efgp_eigenpro_py').is_dir():
            REPO_ROOT = parent
            break
    if REPO_ROOT is not None:
        break

if REPO_ROOT is None:
    raise RuntimeError('Could not find repo root containing efgp_eigenpro_py')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PACKAGE = 'efgp_eigenpro_py.gpu.box_toeplitz_active_block'
MODULE_NAMES = [
    'efgp_eigenpro_py.gpu.versions',
    f'{PACKAGE}.config',
    f'{PACKAGE}.run_experiments',
    f'{PACKAGE}.runner',
    f'{PACKAGE}.box_eigenpro',
    f'{PACKAGE}.diagnostics',
]

def reload_btab_modules():
    modules = {}
    for name in MODULE_NAMES:
        if name in sys.modules:
            modules[name] = importlib.reload(sys.modules[name])
        else:
            modules[name] = importlib.import_module(name)
    return modules

mods = reload_btab_modules()
BTABExperimentConfig = mods[f'{PACKAGE}.config'].BTABExperimentConfig
resolve_btab_experiment_route = mods[f'{PACKAGE}.config'].resolve_btab_experiment_route
run_experiments_module = mods[f'{PACKAGE}.run_experiments']
run_experiments = run_experiments_module.run_experiments
candidate_dataset_stems_for_restore = run_experiments_module.candidate_dataset_stems_for_restore

from efgp_eigenpro_py.gpu.backends import BackendConfig

EXPERIMENT_DIR = REPO_ROOT / 'efgp_eigenpro_py' / 'gpu' / 'box_toeplitz_active_block'
PROCESSED_DATA_DIR = REPO_ROOT / 'efgp_eigenpro_py' / 'gpu' / 'benchmark_dataset' / 'processed'
OUTPUT_ROOT = EXPERIMENT_DIR / 'outputs'
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

def discover_processed_dataset_stems():
    return sorted(p.stem for p in PROCESSED_DATA_DIR.glob('*.npz'))


_STEM_SIZE_SUFFIX_RE = re.compile(r'(?:_ntrain\d+|_n\d+)$', flags=re.IGNORECASE)


def _stem_family_prefix(stem: str) -> str:
    return _STEM_SIZE_SUFFIX_RE.sub('', str(stem).strip())


def resolve_dataset_stems_by_filename(
    dataset_stems: list[str],
    n_train_list: list[int],
) -> list[str]:
    """Match processed/*.npz by filename suffix (_ntrainN / _nN), like report_results."""
    if not n_train_list:
        return [Path(str(s)).stem for s in dataset_stems if str(s).strip()]
    available = set(discover_processed_dataset_stems())
    family_prefixes = list(
        dict.fromkeys(_stem_family_prefix(stem) for stem in dataset_stems if str(stem).strip())
    )
    if not family_prefixes:
        raise ValueError(
            'n_train_list was provided, but no dataset_stems were available to infer dataset families.'
        )
    resolved: list[str] = []
    for prefix in family_prefixes:
        for n_train in n_train_list:
            preferred = [
                f'{prefix}_ntrain{int(n_train)}',
                f'{prefix}_n{int(n_train)}',
            ]
            match = next((name for name in preferred if name in available), None)
            if match is None:
                family_available = sorted(
                    stem for stem in available if _stem_family_prefix(stem) == prefix
                )
                raise FileNotFoundError(
                    f'No processed dataset found for family {prefix!r} with filename scale N={int(n_train):,}. '
                    f'Tried: {", ".join(preferred)}. '
                    f'Available local .npz stems for this family: {", ".join(family_available) or "(none)"}'
                )
            resolved.append(match)
    return list(dict.fromkeys(resolved))


def _copy_file_if_missing(src: Path, dst: Path) -> bool:
    if dst.exists() or not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy2(src, dst)
    return True


def _find_drive_cached_file(filename: str) -> Path | None:
    is_colab = 'google.colab' in sys.modules
    if not is_colab:
        return None
    drive_mount_dir = Path(globals().get('DRIVE_MOUNT_DIR', Path('/content/drive')))
    drive_mydrive_dir = Path(globals().get('DRIVE_MYDRIVE_DIR', drive_mount_dir / 'MyDrive'))
    drive_cache_enabled = bool(globals().get('DRIVE_CACHE_ENABLED', True))
    drive_cache_search_recursive = bool(globals().get('DRIVE_CACHE_SEARCH_RECURSIVE', True))
    drive_cache_candidate_dirs = globals().get(
        'DRIVE_CACHE_CANDIDATE_DIRS',
        [
            drive_mydrive_dir / 'EFGP_Eigenpro' / 'benchmark_dataset_cache',
            drive_mydrive_dir / 'Colab_Experiments' / 'EFGP_Eigenpro' / 'benchmark_dataset_cache',
            drive_mydrive_dir / 'benchmark_dataset_cache',
            drive_mydrive_dir,
        ],
    )
    if not drive_cache_enabled:
        return None
    for base in drive_cache_candidate_dirs:
        base = Path(base)
        if base.exists():
            direct = base / filename
            if direct.exists():
                return direct
    if not drive_cache_search_recursive or not drive_mydrive_dir.exists():
        return None
    try:
        for found in drive_mydrive_dir.rglob(filename):
            if found.is_file():
                return found
    except Exception as exc:
        print(f'Drive cache search skipped for {filename}: {exc}')
    return None


def restore_dataset_from_drive_cache(dataset_stem: str) -> dict:
    restored = {'npz': False, 'json': False, 'from': {}}
    for suffix in ('.npz', '.json'):
        filename = f'{dataset_stem}{suffix}'
        dst = PROCESSED_DATA_DIR / filename
        if dst.exists():
            restored[suffix[1:]] = True
            continue
        src = _find_drive_cached_file(filename)
        if src is None:
            continue
        copied = _copy_file_if_missing(src, dst)
        restored[suffix[1:]] = bool(copied or dst.exists())
        restored['from'][suffix[1:]] = str(src)
        if copied:
            print(f'restored from Drive cache: {dst.name} <- {src}')
    return restored


def ensure_btab_datasets_available(
    dataset_stems,
    *,
    auto_restore: bool = True,
    fail_on_missing: bool = True,
):
    dataset_stems = [Path(str(stem)).stem for stem in dataset_stems]
    available = set(discover_processed_dataset_stems())
    missing = [s for s in dataset_stems if s not in available]
    if missing and auto_restore:
        for stem in missing:
            restore_dataset_from_drive_cache(stem)
        available = set(discover_processed_dataset_stems())
        missing = [s for s in dataset_stems if s not in available]
    if missing:
        message = (
            'Missing processed BTAB datasets: ' + ', '.join(missing) + '\n'
            f'Expected .npz under: {PROCESSED_DATA_DIR}\n'
            'If GitHub only contains .json sidecars, copy the matching .npz files '
            'into processed/ or Google Drive cache first.'
        )
        if fail_on_missing:
            raise FileNotFoundError(message)
        print(message)
    return missing

print('REPO_ROOT =', REPO_ROOT)
print('EXPERIMENT_DIR =', EXPERIMENT_DIR)
print('PROCESSED_DATA_DIR =', PROCESSED_DATA_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)

## 1. 实验配置

`RUN_EXPERIMENT = False` 时会自动加载最新输出目录；设为 `True` 后会重新运行实验。`btab_diagnostic_mode='cheap'` 是默认推荐，专门做谱诊断时再切到 `'full'`。

In [ ]:
RUN_EXPERIMENT = True
LOAD_LATEST_IF_NOT_RUN = True
OUTPUT_PATH = None  # 例如 Path('D:/NU/ML/efgp_eigenpro_py/gpu/box_toeplitz_active_block/outputs/xxx')

AUTO_RESTORE_MISSING_DATASETS = True
FAIL_ON_MISSING_DATASETS = True

DATASET_STEMS = [
    'synthetic_true_func_2d_n1000000',
    'USGS_LPC_IL_Winnebago_2018_ground_elevation_regression_ntrain1000000',
]

EPS_LIST = [1e-5]
SEED = 0

# Kernel sweep. Supported aliases include 'SE'/'rbf'/'gaussian' and 'matern'.
# SE ignores kernel_nu; Matern uses kernel_nu.
KERNEL_FAMILIES = ['SE', 'matern']
KERNEL_LENGTHSCALE = 0.1
KERNEL_NU = 1.5
KERNEL_VARIANCE = 1.0
KERNEL_PARAMS_BY_FAMILY = {
    'SE': {
        'kernel_lengthscale': KERNEL_LENGTHSCALE,
        'kernel_variance': KERNEL_VARIANCE,
    },
    'matern': {
        'kernel_lengthscale': KERNEL_LENGTHSCALE,
        'kernel_nu': KERNEL_NU,
        'kernel_variance': KERNEL_VARIANCE,
    },
}
# 按文件名后缀 _ntrainN / _nN 匹配 processed/*.npz（与 report_results 一致）：
# - 非空时从 DATASET_STEMS 推断 family prefix，再为每个 N 解析具体 stem
# - 留空 [] 则直接使用下方 DATASET_STEMS
N_TRAIN_LIST =[3000_000, 1000_000, 30_000_000, 10_000_000] #[300_000_000, 100_000_000, 30_000_000, 10_000_000]

BACKEND = BackendConfig(xp='cupy', fft='cupy', nufft='auto', linalg='cupy')

SOLVE_TOL = 1e-7
MAXITER = 100000

# Non-Cartesian experiment switch. False restores the legacy Cartesian sweep.
BTAB_CUSTOM = True
#   'custom'   -> use BTAB_CUSTOM_INVERSE_TOPK_LIST and BTAB_CUSTOM_BOXEIG_TOPK_Q_PAIRS
#   'group_a'  -> exact inverse only: topk 1024/2048/4096, dense inverse apply
#   'group_b'  -> boxeig only: topk 4096/8192 x q 128/192
#   'group_c'  -> large-scale 5x-CG shortlist
#   'schedule' -> choose the shortlist automatically from each dataset's n_train
BTAB_CUSTOM_ROUTE = 'group_a'
BTAB_EXPERIMENT_ROUTE = BTAB_CUSTOM_ROUTE if BTAB_CUSTOM else 'cartesian'

# Only used when BTAB_CUSTOM_ROUTE == 'custom'. This is the fully manual
# non-Cartesian route: inverse values and Box-EigenPro pairs are independent.
BTAB_CUSTOM_INVERSE_TOPK_LIST = [1024, 2048, 4096]
BTAB_CUSTOM_BOXEIG_TOPK_Q_PAIRS = [
    (4096, 128),
    (8192, 128),
    (8192, 192),
]

# Active construction. raw active set is expanded to the actual box B.
BTAB_ACTIVE_MODE = 'topk'  # 'topk', 'tau', or 'both' if supported by the local config
BTAB_TOPK_LIST = [256,512,1024,2048,4096]
BTAB_TAU_LIST = [1e-1, 1e-2]
BTAB_BOX_BUDGET = 10000

# Inverse active-block route.
BTAB_SOLVE_MODE = 'auto'
BTAB_EXACT_BOX_MAX_SIZE = 6000
BTAB_EXACT_APPLY_MODE = 'chol_solve' #"inverse" "chol_solve"

# Box-EigenPro route.
BTAB_EIG_Q_LIST = [64,128,192,256]
BTAB_EIG_TOL = 1e-3
BTAB_EIG_MAXITER = None  # implementation defaults to 5*q
BTAB_EIG_NCV = None
BTAB_EIG_APPLY_BATCH_COLS = None

# Diagnostics. Use 'full' for epsilon_T, eta_inv, eta_eig.
BTAB_DIAGNOSTIC_MODE = 'cheap'  # 'none' | 'cheap' | 'full'
DIAGNOSTIC_POWER_ITER = 30
DIAGNOSTIC_TOL = 1e-2

KERNEL_TAG = '_'.join(str(k).strip().lower() for k in KERNEL_FAMILIES) or 'default_kernel'
RUN_TAG = f"btab_{BTAB_EXPERIMENT_ROUTE}_{KERNEL_TAG}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

DATASET_STEMS = [Path(str(s)).stem for s in DATASET_STEMS]
KERNEL_FAMILIES = [str(k).strip() for k in KERNEL_FAMILIES if str(k).strip()]
if not KERNEL_FAMILIES:
    raise ValueError('KERNEL_FAMILIES must contain at least one kernel family.')

cfg = BTABExperimentConfig(
    dataset_stems=DATASET_STEMS,
    eps_list=EPS_LIST,
    seed=SEED,
    kernel_family=KERNEL_FAMILIES[0],
    kernel_family_list=KERNEL_FAMILIES,
    kernel_params_by_family=KERNEL_PARAMS_BY_FAMILY,
    kernel_lengthscale=KERNEL_LENGTHSCALE,
    kernel_nu=KERNEL_NU,
    kernel_variance=KERNEL_VARIANCE,
    n_train_list=N_TRAIN_LIST,
    backend=BACKEND,
    run_tag=RUN_TAG,
    btab_experiment_route=BTAB_EXPERIMENT_ROUTE,
    btab_active_mode=BTAB_ACTIVE_MODE,
    btab_topk_list=BTAB_TOPK_LIST,
    btab_tau_list=BTAB_TAU_LIST,
    btab_inverse_topk_list=(
        BTAB_CUSTOM_INVERSE_TOPK_LIST if BTAB_CUSTOM_ROUTE == 'custom' else None
    ),
    btab_boxeig_topk_q_pairs=(
        BTAB_CUSTOM_BOXEIG_TOPK_Q_PAIRS if BTAB_CUSTOM_ROUTE == 'custom' else None
    ),
    btab_box_budget=BTAB_BOX_BUDGET,
    btab_solve_mode=BTAB_SOLVE_MODE,
    btab_exact_box_max_size=BTAB_EXACT_BOX_MAX_SIZE,
    btab_exact_apply_mode=BTAB_EXACT_APPLY_MODE,
    btab_eig_q_list=BTAB_EIG_Q_LIST,
    btab_eig_tol=BTAB_EIG_TOL,
    btab_eig_maxiter=BTAB_EIG_MAXITER,
    btab_eig_ncv=BTAB_EIG_NCV,
    btab_eig_apply_batch_cols=BTAB_EIG_APPLY_BATCH_COLS,
    btab_diagnostic_mode=BTAB_DIAGNOSTIC_MODE,
    btab_diagnostic_power_iter=DIAGNOSTIC_POWER_ITER,
    btab_diagnostic_tol=DIAGNOSTIC_TOL,
    tol=SOLVE_TOL,
    maxiter=MAXITER,
)

print(json.dumps(asdict(cfg), default=str, indent=2))
print('\nKernel sweep:')
for kernel_family in KERNEL_FAMILIES:
    print(f'  {kernel_family}: {KERNEL_PARAMS_BY_FAMILY.get(kernel_family, {})}')
print('\nResolved BTAB routes:')
for n_train in N_TRAIN_LIST:
    resolved = resolve_btab_experiment_route(cfg, n_train=n_train)
    print(
        f'N={n_train:,}: route={resolved.btab_experiment_route}, '
        f'budget={resolved.btab_box_budget}, '
        f'inverse={resolved.btab_inverse_topk_list}, '
        f'boxeig={resolved.btab_boxeig_topk_q_pairs}'
    )

NameError: name 'BackendConfig' is not defined

In [ ]:
# 先按 N 生成候选 stem，再从 Google Drive / 本地缓存恢复 .npz，最后按文件名后缀 resolve
_resolve_cfg = BTABExperimentConfig(
    dataset_stems=DATASET_STEMS,
    n_train_list=N_TRAIN_LIST,
)
if N_TRAIN_LIST:
    _restore_stems = candidate_dataset_stems_for_restore(_resolve_cfg)
    print('Candidate stems for Drive/local restore:', _restore_stems)
    ensure_btab_datasets_available(
        _restore_stems,
        auto_restore=AUTO_RESTORE_MISSING_DATASETS,
        fail_on_missing=False,
    )
    RESOLVED_DATASET_STEMS = resolve_dataset_stems_by_filename(DATASET_STEMS, N_TRAIN_LIST)
else:
    RESOLVED_DATASET_STEMS = list(DATASET_STEMS)
MISSING_DATASET_STEMS = ensure_btab_datasets_available(
    RESOLVED_DATASET_STEMS,
    auto_restore=AUTO_RESTORE_MISSING_DATASETS,
    fail_on_missing=FAIL_ON_MISSING_DATASETS,
)
AVAILABLE_DATASETS = discover_processed_dataset_stems()
print('Requested DATASET_STEMS (family/explicit):', DATASET_STEMS)
print('N_TRAIN_LIST:', N_TRAIN_LIST)
print('Resolved DATASET_STEMS:', RESOLVED_DATASET_STEMS)
print('Missing DATASET_STEMS after restore:', MISSING_DATASET_STEMS)
print('Available datasets (.npz stems):', AVAILABLE_DATASETS)

# run_experiments 内部也会 resolve；传入显式 stem 并清空 n_train_list，避免 metadata 二次匹配
cfg = replace(cfg, dataset_stems=RESOLVED_DATASET_STEMS, n_train_list=[])

display(pd.DataFrame({'resolved experiment dataset': RESOLVED_DATASET_STEMS}))

## Precomputation Policy

默认 CG baseline 使用项目原始/default precompute。其余实验，包括 EigenPro-PCG、inverse BTAB 和 Box-EigenPro BTAB，统一使用参考 notebook 中的 C1 precompute。这个 patch 只在当前 notebook 进程内生效。

In [ ]:
PRECOMPUTE_POLICY = {
    'plain_cg': 'original',
    'eigenpro_pcg': 'c1',
    'btab_inverse': 'c1',
    'btab_boxeig': 'c1',
}

def install_notebook_precompute_policy(reloaded_mods: dict[str, object]) -> dict[str, str]:
    versions_mod = reloaded_mods['efgp_eigenpro_py.gpu.versions']
    run_mod = reloaded_mods[f'{PACKAGE}.run_experiments']

    bench_mod = importlib.import_module(
        'efgp_eigenpro_py.gpu.benchmark_dataset.accuracy_based_eigenpro_tables'
    )
    bench_mod = importlib.reload(bench_mod)
    bench_cfg = bench_mod.AccuracyBenchmarkConfig(
        precompute_methods={'default': 'c1'},
        binned_quality='balanced',
        binned_use_sparse_bins=False,
        binned_use_gpu_dense_bins=True,
        binned_allow_exact_nufft_fallback=False,
        binned_nufft_allow_cpu_fallback=False,
    )
    bench_mod.install_gpu_precompute_patch(bench_cfg)
    patched_gpu_precompute = bench_mod._gpu_v1_ops_bm.gpu_precompute_v1
    original_run_gpu_precompute = versions_mod._run_gpu_precompute
    versions_mod._NOTEBOOK_PRECOMPUTE_MODE = None

    @contextlib.contextmanager
    def _force_precompute_mode(mode: str):
        previous = getattr(versions_mod, '_NOTEBOOK_PRECOMPUTE_MODE', None)
        versions_mod._NOTEBOOK_PRECOMPUTE_MODE = str(mode).strip().lower()
        try:
            yield
        finally:
            versions_mod._NOTEBOOK_PRECOMPUTE_MODE = previous

    def _run_gpu_precompute_with_policy(
        backend, solver, gpu_cfg, data_ctx, op_ctx, *, use_original_precompute
    ):
        mode = getattr(versions_mod, '_NOTEBOOK_PRECOMPUTE_MODE', None)
        if mode is None:
            return original_run_gpu_precompute(
                backend,
                solver,
                gpu_cfg,
                data_ctx,
                op_ctx,
                use_original_precompute=use_original_precompute,
            )

        previous_active = getattr(bench_mod, '_BENCHMARK_PC_METHOD_ACTIVE', None)
        try:
            bench_mod._BENCHMARK_PC_METHOD_ACTIVE = 'original' if mode == 'original' else 'c1'
            return patched_gpu_precompute(
                backend,
                solver.kernel,
                solver.eps,
                solver.nufft_tol,
                data_ctx,
                op_ctx,
                l2scaled=solver.l2scaled,
                chunk_size=gpu_cfg.chunk_size,
            )
        finally:
            bench_mod._BENCHMARK_PC_METHOD_ACTIVE = previous_active

    def _wrap_runner(fn, mode: str):
        def _wrapped(*args, **kwargs):
            with _force_precompute_mode(mode):
                return fn(*args, **kwargs)
        _wrapped.__name__ = getattr(fn, '__name__', 'wrapped_runner')
        _wrapped.__doc__ = getattr(fn, '__doc__', None)
        return _wrapped

    versions_mod._run_gpu_precompute = _run_gpu_precompute_with_policy
    run_mod.run_v1_pure_efgp = _wrap_runner(
        run_mod.run_v1_pure_efgp, PRECOMPUTE_POLICY['plain_cg']
    )
    run_mod.run_v3_full_gpu_eigenspace = _wrap_runner(
        run_mod.run_v3_full_gpu_eigenspace, PRECOMPUTE_POLICY['eigenpro_pcg']
    )
    run_mod.run_v6_box_toeplitz_active_block = _wrap_runner(
        run_mod.run_v6_box_toeplitz_active_block, PRECOMPUTE_POLICY['btab_inverse']
    )
    run_mod.run_v7_box_eigenpro_active_block = _wrap_runner(
        run_mod.run_v7_box_eigenpro_active_block, PRECOMPUTE_POLICY['btab_boxeig']
    )
    return dict(PRECOMPUTE_POLICY)

print('Notebook precompute policy prepared:', PRECOMPUTE_POLICY)

## 2. 运行或加载实验结果

In [ ]:
def latest_output_dir(root: Path) -> Path:
    if not root.exists():
        raise FileNotFoundError(f'Output root does not exist: {root}')
    candidates = [
        p for p in root.iterdir()
        if p.is_dir() and any((p / name).exists() for name in [
            'master_summary.csv', 'aggregate_summary.csv', 'btab_master_summary.csv', 'btab_aggregate_summary.csv'
        ])
    ]
    if not candidates:
        raise FileNotFoundError(f'No experiment output directories found under {root}')
    return max(candidates, key=lambda p: p.stat().st_mtime)

if RUN_EXPERIMENT:
    mods = reload_btab_modules()
    policy_info = install_notebook_precompute_policy(mods)
    run_experiments_module = mods[f'{PACKAGE}.run_experiments']
    run_experiments = run_experiments_module.run_experiments
    print('Precompute policy:', policy_info)
    result = run_experiments(cfg)
    OUTPUT_PATH = Path(result['output_dir']).resolve()
elif OUTPUT_PATH is not None:
    OUTPUT_PATH = Path(OUTPUT_PATH).resolve()
elif LOAD_LATEST_IF_NOT_RUN:
    OUTPUT_PATH = latest_output_dir(OUTPUT_ROOT)
else:
    raise RuntimeError('Set RUN_EXPERIMENT=True, OUTPUT_PATH=..., or LOAD_LATEST_IF_NOT_RUN=True')

print('OUTPUT_PATH =', OUTPUT_PATH)

In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        print(f'loaded {path.name}: {df.shape}')
        return df
    print(f'missing {path.name}')
    return pd.DataFrame()

master_df = read_csv_if_exists(OUTPUT_PATH / 'master_summary.csv')
aggregate_df = read_csv_if_exists(OUTPUT_PATH / 'aggregate_summary.csv')
btab_master_df = read_csv_if_exists(OUTPUT_PATH / 'btab_master_summary.csv')
btab_aggregate_df = read_csv_if_exists(OUTPUT_PATH / 'btab_aggregate_summary.csv')

# master tables include plain CG and EigenPro baselines; btab tables only contain BTAB routes.
result_df = master_df if not master_df.empty else btab_master_df
summary_df = aggregate_df if not aggregate_df.empty else btab_aggregate_df

if result_df.empty:
    raise RuntimeError('No result table found. Run experiments first or set OUTPUT_PATH to a valid output directory.')

display(result_df)

## 3. 整理字段

这里把两条路线统一标成 `route = inverse | boxeig`，并提取 selector tag，避免 raw active set 和 expanded box 混在一起看。

In [ ]:
def existing_cols(df: pd.DataFrame, cols: list[str]) -> list[str]:
    return [c for c in cols if c in df.columns]

def display_cols(df: pd.DataFrame, cols: list[str], sort_by: list[str] | None = None, n: int | None = None):
    cols = existing_cols(df, cols)
    out = df.copy()
    if sort_by:
        sort_cols = existing_cols(out, sort_by)
        if sort_cols:
            out = out.sort_values(sort_cols)
    shown = out[cols] if n is None else out[cols].head(n)
    display(shown)
    return out[cols]

def classify_route(method: str) -> str:
    method = str(method)
    if method == 'plain_cg':
        return 'cg_baseline'
    if method.startswith('eigenpro_pcg'):
        return 'eigenpro_baseline'
    if 'boxeig' in method:
        return 'boxeig'
    if method.startswith('btab'):
        return 'inverse'
    return 'other'

def selector_tag(method: str) -> str:
    method = str(method)
    tag = re.sub(r'^btab_boxeig_', '', method)
    tag = re.sub(r'^btab_(auto|exact|inner_pcg)_', '', tag)
    tag = re.sub(r'_q\d+$', '', tag)
    return tag

work_df = result_df.copy()
work_df['method'] = work_df['method'].astype(str)
work_df['route'] = work_df['method'].map(classify_route)
work_df['selector_tag'] = work_df['method'].map(selector_tag)
work_df['precompute_mode'] = work_df['route'].map({
    'cg_baseline': 'original/default',
    'eigenpro_baseline': 'c1',
    'inverse': 'c1',
    'boxeig': 'c1',
}).fillna('unknown')

for col in work_df.columns:
    if col not in ['dataset_stem', 'method', 'route', 'selector_tag', 'precompute_mode', 'S_kind', 'btab_eig_backend', 'btab_diagnostic_mode']:
        work_df[col] = pd.to_numeric(work_df[col], errors='ignore')

cg_baseline_df = work_df[work_df['route'] == 'cg_baseline'].copy()
eigenpro_baseline_df = work_df[work_df['route'] == 'eigenpro_baseline'].copy()
inverse_df = work_df[work_df['route'] == 'inverse'].copy()
boxeig_df = work_df[work_df['route'] == 'boxeig'].copy()

print('all experiment rows:', len(work_df))
print('plain CG baseline rows:', len(cg_baseline_df))
print('EigenPro baseline rows:', len(eigenpro_baseline_df))
print('inverse rows:', len(inverse_df))
print('boxeig rows:', len(boxeig_df))

## 4. Baseline 与共同诊断

### 4.1 Candidate B/R 与共同诊断

`btab_active_size_raw` 是 threshold/topk 得到的 raw active set 大小；`btab_box_size` 是 expanded box `B` 的大小。preconditioner 和 diagnostics 应优先看 `btab_box_size`。

In [ ]:
common_cols = [
    'dataset_stem', 'seed', 'eps', 'method', 'route', 'precompute_mode',
    'selector_tag', 'S_kind',
    'btab_active_size_raw', 'btab_box_size', 'active_size', 'box_size', 'tail_size',
    'rho_max_T', 'tail_energy', 'epsilon_T', 'btab_diagnostic_mode',
    'time_post_diagnostics', 'time_total_with_diagnostics',
    'diagnostic_n_matvec', 'diagnostic_n_A_matvec', 'diagnostic_n_ASS_matvec',
    'diagnostic_n_precond', 'diagnostic_n_eig_matvec', 'diagnostic_n_power_iter',
]

btab_routes_df = work_df[work_df['route'].isin(['inverse', 'boxeig'])].copy()
candidate_common = display_cols(
    btab_routes_df,
    common_cols,
    sort_by=['dataset_stem', 'seed', 'eps', 'selector_tag', 'route', 'btab_eig_q'],
)

### 4.2 Baseline：默认 CG 与 EigenPro-PCG

`plain_cg` 是默认 CG baseline，使用 original/default precompute；EigenPro-PCG baseline 使用 C1 precompute，与 inverse BTAB 和 Box-EigenPro BTAB 保持一致。

In [ ]:
baseline_cols = [
    'dataset_stem', 'n_train', 'eps', 'method', 'route', 'precompute_mode', 'top_q',
    'iterations', 'cg_iters', 'n_matvec', 'cg_n_matvec', 'n_precond',
    'time_precompute', 'time_eigenspace', 'time_precond_build', 'time_solve',
    'time_predict', 'time_total', 'rmse_train', 'rmse_test', 'cg_relres', 'status',
]
baseline_df = work_df[
    work_df['route'].isin(['cg_baseline', 'eigenpro_baseline'])
].copy()
baseline_table = display_cols(
    baseline_df,
    baseline_cols,
    sort_by=['dataset_stem', 'n_train', 'eps', 'route', 'top_q'],
)

## 5. Inverse Active-Block Route

full diagnostics 模式下这里会出现 `epsilon_T` 和 `eta_inv`。inverse route 的 `eta_inv` 复用 build 阶段已有 Cholesky / inverse apply，不重复 factorization。

In [ ]:
inverse_cols = [
    'dataset_stem', 'seed', 'eps', 'method', 'precompute_mode', 'selector_tag',
    'btab_active_size_raw', 'btab_box_size', 'tail_size',
    'rho_max_T', 'tail_energy', 'epsilon_T', 'eta_inv', 'eta_inv_sq', 'eta_inv_status',
    'cg_iters', 'outer_iters', 'time_precond_build', 'time_solve', 'time_total',
    'time_post_diagnostics', 'time_total_with_diagnostics',
    'rmse_train', 'rmse_test', 'relres_final',
]

inverse_table = display_cols(
    inverse_df,
    inverse_cols,
    sort_by=['dataset_stem', 'seed', 'eps', 'selector_tag'],
)

## 6. Box-EigenPro Route

`btab_eig_theta_q1_over_sigma2` 判断 EigenPro 是否把 `B` block 压平；`btab_eig_eig_residual_max` 检查 eigenpairs 质量；full diagnostics 模式下额外看 `eta_eig`。

In [ ]:
boxeig_cols = [
    'dataset_stem', 'seed', 'eps', 'method', 'precompute_mode',
    'selector_tag', 'btab_eig_q',
    'btab_active_size_raw', 'btab_box_size', 'btab_eig_size_S', 'btab_eig_size_T',
    'rho_max_T', 'tail_energy', 'epsilon_T',
    'btab_eig_theta_q1', 'btab_eig_theta_q1_over_sigma2', 'theta_q1_over_sigma2',
    'btab_eig_eig_residual_max', 'btab_eig_eig_residual_median',
    'eta_eig', 'eta_eig_sq',
    'btab_eig_storage_bytes', 'btab_eig_n_eig_matvec', 'btab_eig_n_ABB_matvec_cols',
    'btab_eig_ncv_actual', 'btab_eig_backend',
    'cg_iters', 'outer_iters', 'time_eig_setup', 'time_precond_build', 'time_solve', 'time_total',
    'time_post_diagnostics', 'time_total_with_diagnostics',
    'rmse_train', 'rmse_test', 'relres_final',
]

boxeig_table = display_cols(
    boxeig_df,
    boxeig_cols,
    sort_by=['dataset_stem', 'seed', 'eps', 'selector_tag', 'btab_eig_q'],
)

## 7. 两条路线对比

这个表按 dataset / seed / eps / selector 把 inverse 与 boxeig 放在一起。boxeig 会保留不同 `q` 的 sensitivity。

In [ ]:
comparison_cols = [
    'dataset_stem', 'n_train', 'seed', 'eps', 'selector_tag', 'route', 'method',
    'precompute_mode', 'top_q', 'btab_eig_q',
    'btab_active_size_raw', 'btab_box_size', 'tail_size',
    'rho_max_T', 'tail_energy', 'epsilon_T', 'eta_inv', 'eta_eig',
    'btab_eig_theta_q1_over_sigma2', 'btab_eig_eig_residual_max',
    'cg_iters', 'outer_iters', 'time_precond_build', 'time_solve', 'time_total',
    'time_post_diagnostics', 'time_total_with_diagnostics', 'rmse_test', 'relres_final',
]

route_comparison = display_cols(
    work_df,
    comparison_cols,
    sort_by=['dataset_stem', 'seed', 'eps', 'selector_tag', 'route', 'btab_eig_q'],
)

## 8. 图：时间、迭代数和谱诊断

In [ ]:
plot_df = work_df.copy()
if 'btab_box_size' not in plot_df.columns and 'box_size' in plot_df.columns:
    plot_df['btab_box_size'] = plot_df['box_size']
if 'cg_iters' not in plot_df.columns and 'outer_iters' in plot_df.columns:
    plot_df['cg_iters'] = plot_df['outer_iters']

def label_for_row(row):
    if row['route'] == 'cg_baseline':
        return 'plain CG'
    if row['route'] == 'eigenpro_baseline':
        q = row.get('top_q', np.nan)
        return f'EigenPro-PCG q={int(q)}' if pd.notna(q) else 'EigenPro-PCG'
    if row['route'] == 'boxeig':
        q = row.get('btab_eig_q', np.nan)
        return f'boxeig q={int(q)}' if pd.notna(q) else 'boxeig'
    return 'inverse BTAB'

if not plot_df.empty and 'btab_box_size' in plot_df.columns:
    plot_df['plot_label'] = plot_df.apply(label_for_row, axis=1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

    for label, part in plot_df.groupby('plot_label'):
        part = part.sort_values('btab_box_size')
        if 'cg_iters' in part.columns:
            axes[0].plot(part['btab_box_size'], part['cg_iters'], marker='o', label=label)
        if 'time_total' in part.columns:
            axes[1].plot(part['btab_box_size'], part['time_total'], marker='o', label=label)
        y_col = 'time_total_with_diagnostics' if 'time_total_with_diagnostics' in part.columns else 'time_total'
        if y_col in part.columns:
            axes[2].plot(part['btab_box_size'], part[y_col], marker='o', label=label)

    axes[0].set_title('CG iterations')
    axes[1].set_title('time_total')
    axes[2].set_title('time_total_with_diagnostics')
    for ax in axes:
        ax.set_xlabel('expanded box size |B|')
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

diag_metrics = [
    ('rho_max_T', 'rho_max_T'),
    ('epsilon_T', 'epsilon_T'),
    ('btab_eig_theta_q1_over_sigma2', 'theta_q1 / sigma2'),
    ('btab_eig_eig_residual_max', 'eig residual max'),
    ('eta_inv', 'eta_inv'),
    ('eta_eig', 'eta_eig'),
]
available_metrics = [(c, title) for c, title in diag_metrics if c in plot_df.columns]
if available_metrics:
    ncols = 3
    nrows = int(np.ceil(len(available_metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.2 * nrows), squeeze=False)
    for ax, (col, title) in zip(axes.ravel(), available_metrics):
        for label, part in plot_df.groupby('plot_label'):
            part = part.sort_values('btab_box_size')
            if part[col].notna().any():
                ax.plot(part['btab_box_size'], part[col], marker='o', label=label)
        ax.set_title(title)
        ax.set_xlabel('expanded box size |B|')
        ax.grid(True, alpha=0.3)
        ax.legend()
    for ax in axes.ravel()[len(available_metrics):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

### 8.1 最佳配置随 N 的变化

默认以不包含 post diagnostics 的 `time_total` 为效果指标，数值越小越好。重复实验先按配置取中位数，再分别选择 inverse BTAB 的最佳 top-k，以及 Box-EigenPro 的最佳 `(top-k, q)`。可把 `BEST_CONFIG_METRIC` 改成 `cg_iters`、`outer_iters` 或其他数值字段。

In [ ]:
BEST_CONFIG_METRIC = 'time_total'
BEST_CONFIG_ASCENDING = True
BEST_CONFIG_REQUIRE_OK = True

def dataset_family_from_stem(stem: str) -> str:
    return re.sub(r'(?:_ntrain\d+|_n\d+)$', '', str(stem), flags=re.IGNORECASE)

def numeric_from_column_or_method(df: pd.DataFrame, column: str, pattern: str) -> pd.Series:
    values = pd.Series(np.nan, index=df.index, dtype=float)
    if column in df.columns:
        values = pd.to_numeric(df[column], errors='coerce')
    if 'method' not in df.columns:
        return values
    parsed = pd.to_numeric(df['method'].astype(str).str.extract(pattern, expand=False), errors='coerce')
    return values.fillna(parsed)

def empty_best_config_table(include_q: bool) -> pd.DataFrame:
    cols = ['dataset_family', 'n_train', 'eps', 'selected_topk', BEST_CONFIG_METRIC]
    if include_q:
        cols.insert(4, 'selected_q')
    return pd.DataFrame(columns=cols)

def select_best_configs(df: pd.DataFrame, *, include_q: bool) -> pd.DataFrame:
    candidates = df.copy()
    if candidates.empty:
        return empty_best_config_table(include_q)
    if BEST_CONFIG_REQUIRE_OK and 'status' in candidates.columns:
        candidates = candidates[
            candidates['status'].fillna('ok').astype(str).str.lower().eq('ok')
        ]
    if BEST_CONFIG_METRIC not in candidates.columns:
        print(f'Best-config metric is missing for this route: {BEST_CONFIG_METRIC}')
        return empty_best_config_table(include_q)

    candidates[BEST_CONFIG_METRIC] = pd.to_numeric(
        candidates[BEST_CONFIG_METRIC], errors='coerce'
    )
    candidates['n_train'] = pd.to_numeric(candidates['n_train'], errors='coerce')
    candidates['dataset_family'] = candidates['dataset_stem'].map(dataset_family_from_stem)
    candidates['selected_topk'] = numeric_from_column_or_method(
        candidates, 'btab_active_topk', r'topk_(\d+)'
    )
    config_cols = ['dataset_family', 'n_train', 'eps', 'selected_topk']
    required = ['n_train', 'selected_topk', BEST_CONFIG_METRIC]
    if include_q:
        candidates['selected_q'] = numeric_from_column_or_method(
            candidates, 'btab_eig_q', r'_q(\d+)$'
        )
        config_cols.append('selected_q')
        required.append('selected_q')
    candidates = candidates.dropna(subset=required)

    value_cols = existing_cols(
        candidates,
        [
            BEST_CONFIG_METRIC, 'iterations', 'cg_iters', 'outer_iters', 'rmse_test',
            'btab_eig_theta_q1_over_sigma2', 'btab_eig_eig_residual_max', 'eta_eig',
        ],
    )
    summary = (
        candidates.groupby(config_cols, as_index=False)[value_cols]
        .median(numeric_only=True)
    )
    if summary.empty:
        return summary
    chooser = summary.groupby(['dataset_family', 'n_train', 'eps'])[BEST_CONFIG_METRIC]
    best_idx = chooser.idxmin() if BEST_CONFIG_ASCENDING else chooser.idxmax()
    return summary.loc[best_idx].sort_values(
        ['dataset_family', 'eps', 'n_train']
    ).reset_index(drop=True)

best_inverse_by_n = select_best_configs(inverse_df, include_q=False)
best_boxeig_by_n = select_best_configs(boxeig_df, include_q=True)

print('Best inverse BTAB by N')
display(best_inverse_by_n)
print('Best Box-EigenPro BTAB by N')
display(best_boxeig_by_n)

### 8.2 可视化一：Inverse BTAB 最佳 top-k 与 N

In [ ]:
inverse_families = list(best_inverse_by_n['dataset_family'].drop_duplicates())
if inverse_families:
    fig, axes = plt.subplots(
        2, len(inverse_families), figsize=(7 * len(inverse_families), 8), squeeze=False
    )
    for col_idx, family in enumerate(inverse_families):
        family_df = best_inverse_by_n[best_inverse_by_n['dataset_family'] == family]
        for eps_value, part in family_df.groupby('eps', dropna=False):
            part = part.sort_values('n_train')
            label = f'eps={eps_value:g}' if pd.notna(eps_value) else 'eps=NA'
            axes[0, col_idx].plot(
                part['n_train'], part['selected_topk'], marker='o', linewidth=2, label=label
            )
            axes[1, col_idx].plot(
                part['n_train'], part[BEST_CONFIG_METRIC], marker='o', linewidth=2, label=label
            )
            for row in part.itertuples(index=False):
                axes[0, col_idx].annotate(
                    f'k={int(row.selected_topk)}',
                    (row.n_train, row.selected_topk),
                    xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9,
                )
        axes[0, col_idx].set_title(f'{family}\nBest inverse BTAB top-k')
        axes[0, col_idx].set_ylabel('selected top-k')
        axes[1, col_idx].set_title(f'Selected {BEST_CONFIG_METRIC}')
        axes[1, col_idx].set_ylabel(BEST_CONFIG_METRIC)
        for row_idx in range(2):
            axes[row_idx, col_idx].set_xscale('log')
            axes[row_idx, col_idx].set_xlabel('N train')
            axes[row_idx, col_idx].grid(True, alpha=0.3)
            axes[row_idx, col_idx].legend()
    plt.tight_layout()
    plt.show()
else:
    print('No successful inverse BTAB candidates are available for the selected metric.')

### 8.3 可视化二：Box-EigenPro 最佳 (top-k, q) 与 N

In [ ]:
boxeig_families = list(best_boxeig_by_n['dataset_family'].drop_duplicates())
if boxeig_families:
    fig, axes = plt.subplots(
        2, len(boxeig_families), figsize=(7 * len(boxeig_families), 8), squeeze=False
    )
    for col_idx, family in enumerate(boxeig_families):
        family_df = best_boxeig_by_n[best_boxeig_by_n['dataset_family'] == family]
        q_axis = axes[0, col_idx].twinx()
        for eps_value, part in family_df.groupby('eps', dropna=False):
            part = part.sort_values('n_train')
            label = f'eps={eps_value:g}' if pd.notna(eps_value) else 'eps=NA'
            topk_line = axes[0, col_idx].plot(
                part['n_train'], part['selected_topk'], marker='o', linewidth=2,
                label=f'top-k, {label}',
            )
            q_axis.plot(
                part['n_train'], part['selected_q'], marker='s', linestyle='--', linewidth=2,
                color=topk_line[0].get_color(), label=f'q, {label}',
            )
            axes[1, col_idx].plot(
                part['n_train'], part[BEST_CONFIG_METRIC], marker='o', linewidth=2, label=label
            )
            for row in part.itertuples(index=False):
                axes[0, col_idx].annotate(
                    f'k={int(row.selected_topk)}, q={int(row.selected_q)}',
                    (row.n_train, row.selected_topk),
                    xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9,
                )
        axes[0, col_idx].set_title(f'{family}\nBest Box-EigenPro configuration')
        axes[0, col_idx].set_ylabel('selected top-k')
        q_axis.set_ylabel('selected q')
        lines_a, labels_a = axes[0, col_idx].get_legend_handles_labels()
        lines_b, labels_b = q_axis.get_legend_handles_labels()
        axes[0, col_idx].legend(lines_a + lines_b, labels_a + labels_b, loc='best')
        axes[1, col_idx].set_title(f'Selected {BEST_CONFIG_METRIC}')
        axes[1, col_idx].set_ylabel(BEST_CONFIG_METRIC)
        axes[0, col_idx].set_xscale('log')
        q_axis.set_xscale('log')
        axes[1, col_idx].set_xscale('log')
        axes[0, col_idx].set_xlabel('N train')
        axes[1, col_idx].set_xlabel('N train')
        axes[0, col_idx].grid(True, alpha=0.3)
        axes[1, col_idx].grid(True, alpha=0.3)
        axes[1, col_idx].legend()
    plt.tight_layout()
    plt.show()
else:
    print('No successful Box-EigenPro candidates are available for the selected metric.')

## 9. 导出整理后的表格

In [ ]:
EXPORT_TABLES = True

if EXPORT_TABLES:
    export_dir = OUTPUT_PATH / 'diagnostic_tables'
    export_dir.mkdir(parents=True, exist_ok=True)

    tables = {
        'baseline_cg_eigenpro.csv': baseline_table,
        'candidate_B_common_diagnostics.csv': candidate_common,
        'inverse_diagnostics.csv': inverse_table,
        'boxeig_diagnostics.csv': boxeig_table,
        'route_comparison.csv': route_comparison,
        'best_inverse_by_n.csv': best_inverse_by_n,
        'best_boxeig_by_n.csv': best_boxeig_by_n,
    }
    for name, table in tables.items():
        path = export_dir / name
        table.to_csv(path, index=False)
        print('wrote', path)

### 9.1 Table 1 制表集导出

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

TABLE1_DATASET_MODE = globals().get('TABLE1_DATASET_MODE', 'synthetic')
TABLE1_RMSE_REL_TOL = globals().get('TABLE1_RMSE_REL_TOL', 0.10)
TABLE1_MASTER_PATHS = globals().get('TABLE1_MASTER_PATHS', None)

TABLE1_METHOD_ORDER = {
    'EFGP-CG': 0,
    'EigenPro-PCG': 1,
    'Active inverse': 2,
    'Box-EigenPro': 3,
}


def method_family(method: str) -> str:
    method = str(method)
    if method == 'plain_cg':
        return 'EFGP-CG'
    if re.fullmatch(r'eigenpro_pcg_q\d+', method):
        return 'EigenPro-PCG'
    if re.fullmatch(r'btab_auto_topk_\d+', method):
        return 'Active inverse'
    if re.fullmatch(r'btab_boxeig_topk_\d+_q\d+', method):
        return 'Box-EigenPro'
    return 'Other'


def kernel_label(row) -> str:
    fam = str(row.get('kernel_family', '')).lower()
    if fam in {'matern', 'mat\u00e9rn'}:
        nu = row.get('kernel_nu', np.nan)
        if pd.notna(nu):
            return f'Matern-{nu:g}'
        return 'Matern'
    if fam in {'se', 'sqexp', 'squared_exponential', 'gaussian'}:
        return 'SE'
    return str(row.get('kernel_family', ''))


def parse_topk(method: str):
    m = re.search(r'topk_(\d+)', str(method))
    return int(m.group(1)) if m else np.nan


def parse_q(method: str):
    m = re.search(r'_q(\d+)$', str(method))
    return int(m.group(1)) if m else np.nan


def table1_master_paths() -> list[Path]:
    if TABLE1_MASTER_PATHS:
        return [Path(p).resolve() for p in TABLE1_MASTER_PATHS]
    return [(OUTPUT_PATH / 'master_summary.csv').resolve()]


def load_table1_master() -> pd.DataFrame:
    frames = []
    for path in table1_master_paths():
        if not path.exists():
            print('missing Table 1 source:', path)
            continue
        frame = pd.read_csv(path)
        frame['table1_source'] = path.parent.name
        frames.append(frame)
        print(f'loaded Table 1 source {path}: {frame.shape}')
    if not frames:
        raise FileNotFoundError('No Table 1 master_summary.csv sources were found.')
    return pd.concat(frames, ignore_index=True, sort=False)


def filter_table1_dataset_mode(df: pd.DataFrame) -> pd.DataFrame:
    if TABLE1_DATASET_MODE != 'synthetic' or 'dataset_stem' not in df.columns:
        return df.copy()
    stem = df['dataset_stem'].fillna('').astype(str).str.lower()
    synthetic_mask = stem.str.contains(r'synthetic|true_func', regex=True)
    if synthetic_mask.any():
        return df[synthetic_mask].copy()
    real_markers = r'protein|year|higgs|susy|covtype|mnist|fashion|cifar|real|uci|airline|taxi'
    return df[~stem.str.contains(real_markers, regex=True)].copy()


def ensure_table1_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col not in df.columns:
            df[col] = np.nan
    return df


def add_table1_speedup(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    baseline = out[out['method_family'] == 'EFGP-CG'].copy()
    if baseline.empty:
        out['efgp_time_total'] = np.nan
        out['Speedup'] = np.nan
        return out
    baseline = baseline.sort_values('time_total').groupby(group_cols, dropna=False).head(1)
    baseline = baseline[group_cols + ['time_total']].rename(columns={'time_total': 'efgp_time_total'})
    out = out.merge(baseline, on=group_cols, how='left')
    out['Speedup'] = out['efgp_time_total'] / out['time_total']
    return out


def prepare_table1_candidates(master_df: pd.DataFrame) -> pd.DataFrame:
    df = filter_table1_dataset_mode(master_df).copy()
    df = ensure_table1_columns(
        df,
        [
            'dataset_stem', 'n_train', 'kernel_family', 'kernel_nu', 'kernel_lengthscale',
            'eps', 'reg_lambda', 'method', 'M', 'mtot', 'btab_active_topk',
            'btab_box_size', 'btab_eig_q', 'cg_iters', 'time_solve', 'time_total',
            'rmse_test', 'rmse_train', 'status', 'is_warmup',
        ],
    )
    df['method'] = df['method'].astype(str)
    df['method_family'] = df['method'].map(method_family)
    df = df[df['method_family'].isin(TABLE1_METHOD_ORDER)].copy()
    df = df[df['status'].fillna('ok').astype(str).str.lower().eq('ok')].copy()
    df = df[~df['is_warmup'].fillna(False).astype(bool)].copy()

    numeric_cols = [
        'n_train', 'kernel_nu', 'kernel_lengthscale', 'eps', 'reg_lambda', 'M', 'mtot',
        'btab_active_topk', 'btab_box_size', 'btab_eig_q', 'cg_iters', 'time_solve',
        'time_total', 'rmse_test', 'rmse_train',
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['kernel_label'] = df.apply(kernel_label, axis=1)
    df['kernel'] = df['kernel_label']
    df['topk'] = df['method'].map(parse_topk)
    df['q'] = df['btab_eig_q'].fillna(df['method'].map(parse_q))
    # reg_lambda is the solver ridge/noise regularization; keep it for paper reproducibility.
    group_cols = ['dataset_stem', 'n_train', 'kernel_label', 'kernel_lengthscale', 'eps', 'reg_lambda']
    df = add_table1_speedup(df, group_cols)

    keep_cols = [
        'dataset_stem', 'n_train', 'kernel', 'kernel_family', 'kernel_label',
        'kernel_nu', 'kernel_lengthscale', 'eps', 'reg_lambda', 'method',
        'method_family', 'M', 'mtot', 'topk', 'q', 'btab_active_topk',
        'btab_box_size', 'btab_eig_q', 'cg_iters', 'time_solve', 'time_total',
        'rmse_test', 'rmse_train', 'Speedup', 'status', 'table1_source',
    ]
    return df[[c for c in keep_cols if c in df.columns]].copy()


def make_table1_placeholder(group_values: tuple, group_cols: list[str], family: str) -> pd.DataFrame:
    row = dict(zip(group_cols, group_values))
    row.update({
        'kernel': row.get('kernel_label', np.nan),
        'method': '--',
        'method_family': family,
        'selection_note': f'no candidate with rmse_train <= {1.0 + TABLE1_RMSE_REL_TOL:.2f} * EFGP-CG',
    })
    return pd.DataFrame([row])


def select_table1_rows(candidates: pd.DataFrame) -> pd.DataFrame:
    if candidates.empty:
        return pd.DataFrame()
    group_cols = ['dataset_stem', 'n_train', 'kernel_label', 'kernel_lengthscale', 'eps', 'reg_lambda']
    selected = []

    for group_values, part in candidates.groupby(group_cols, dropna=False):
        base = part[part['method_family'] == 'EFGP-CG'].copy()
        base = base.dropna(subset=['time_total', 'rmse_train'])
        if base.empty:
            continue
        base = base.sort_values('time_total').iloc[[0]].copy()
        base_rmse = float(base['rmse_train'].iloc[0])
        base_total = float(base['time_total'].iloc[0])
        base['Speedup'] = 1.0
        base['selection_note'] = 'EFGP-CG baseline; speedup uses time_total'
        selected.append(base)

        for family in ['EigenPro-PCG', 'Active inverse', 'Box-EigenPro']:
            cand = part[part['method_family'] == family].copy()
            cand = cand.dropna(subset=['time_total', 'rmse_train'])
            cand = cand[cand['rmse_train'] <= (1.0 + TABLE1_RMSE_REL_TOL) * base_rmse].copy()
            if cand.empty:
                selected.append(make_table1_placeholder(group_values, group_cols, family))
                continue
            best = cand.sort_values('time_total').iloc[[0]].copy()
            best['Speedup'] = base_total / float(best['time_total'].iloc[0])
            best['selection_note'] = 'selected by min time_total under 1.1x EFGP-CG train RMSE'
            selected.append(best)

    if not selected:
        return pd.DataFrame()
    out = pd.concat(selected, ignore_index=True, sort=False)
    out['method_order'] = out['method_family'].map(TABLE1_METHOD_ORDER)
    out = out.sort_values(['kernel_label', 'n_train', 'method_order']).drop(columns=['method_order'])
    return out


def make_table1_sweep_appendix(candidates: pd.DataFrame) -> pd.DataFrame:
    if candidates.empty:
        return pd.DataFrame()
    group_cols = ['dataset_stem', 'n_train', 'kernel_label', 'kernel_lengthscale', 'eps', 'reg_lambda']
    sweep = candidates[candidates['method_family'].isin(['Active inverse', 'Box-EigenPro'])].copy()
    if sweep.empty:
        return sweep
    baseline = candidates[candidates['method_family'] == 'EFGP-CG'].copy()
    baseline = baseline.sort_values('time_total').groupby(group_cols, dropna=False).head(1)
    baseline = baseline[group_cols + ['rmse_train', 'time_total']].rename(
        columns={'rmse_train': 'RMSE_EFGP', 'time_total': 'T_total_EFGP'}
    )
    sweep = sweep.merge(baseline, on=group_cols, how='left')
    sweep['RMSE / RMSE_EFGP'] = sweep['rmse_train'] / sweep['RMSE_EFGP']
    sweep['Speedup'] = sweep['T_total_EFGP'] / sweep['time_total']
    out = pd.DataFrame({
        'N': sweep['n_train'],
        'Kernel': sweep['kernel_label'],
        'reg_lambda': sweep['reg_lambda'],
        'Method': sweep['method_family'],
        'method': sweep['method'],
        'top-k': sweep['topk'],
        '|B|': sweep['btab_box_size'],
        'q': sweep['q'],
        'Iter.': sweep['cg_iters'],
        'T_solve': sweep['time_solve'],
        'T_total': sweep['time_total'],
        'RMSE': sweep['rmse_train'],
        'RMSE / RMSE_EFGP': sweep['RMSE / RMSE_EFGP'],
        'Speedup': sweep['Speedup'],
    })
    return out.sort_values(['Kernel', 'N', 'Method', 'top-k', 'q']).reset_index(drop=True)


table1_master_df = load_table1_master()
paper_table1_candidates = prepare_table1_candidates(table1_master_df)
paper_table1_selected = select_table1_rows(paper_table1_candidates)
paper_table1_sweep_appendix = make_table1_sweep_appendix(paper_table1_candidates)

export_dir = OUTPUT_PATH / 'diagnostic_tables'
export_dir.mkdir(parents=True, exist_ok=True)
for name, table in {
    'paper_table1_candidates.csv': paper_table1_candidates,
    'paper_table1_selected.csv': paper_table1_selected,
    'paper_table1_sweep_appendix.csv': paper_table1_sweep_appendix,
}.items():
    path = export_dir / name
    table.to_csv(path, index=False)
    print('wrote', path, table.shape)

print('Table 1 dataset mode:', TABLE1_DATASET_MODE)
print('Table 1 RMSE relative tolerance:', TABLE1_RMSE_REL_TOL)
display(paper_table1_selected)

In [ ]:
# ---- Optional Colab export / download / disconnect ----
# 在 Colab 中把当前 run 的输出、当前 notebook 和 manifest 打成 zip，
# 同步到 Google Drive，并可触发浏览器下载，最后按需自动断开 runtime。
import json as _json
import shutil as _shutil
import time as _time
from datetime import timedelta
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
ENABLE_COLAB_EXPORT = True

if not (IS_COLAB and ENABLE_COLAB_EXPORT):
    print("[note] export cell skipped (IS_COLAB=", IS_COLAB, ", ENABLE_COLAB_EXPORT=", ENABLE_COLAB_EXPORT, ")")
else:
    try:
        from google.colab import drive, files, runtime
        _HAS_COLAB_EXPORT = True
    except Exception:
        drive = None
        files = None
        runtime = None
        _HAS_COLAB_EXPORT = False

    end_time = _time.time()
    try:
        elapsed_total = end_time - start_time
        time_str = str(timedelta(seconds=int(elapsed_total)))
    except NameError:
        time_str = "unknown (start_time was not set)"

    if "OUTPUT_PATH" in globals() and OUTPUT_PATH is not None:
        out_dir_path = Path(OUTPUT_PATH).resolve()
    else:
        if "OUT_ROOT" in globals():
            candidate_out_root = Path(OUT_ROOT).resolve()
        elif "REPO_ROOT" in globals():
            candidate_out_root = (
                Path(REPO_ROOT).resolve()
                / "efgp_eigenpro_py"
                / "gpu"
                / "box_toeplitz_active_block"
                / "outputs"
            )
        else:
            candidate_out_root = Path.cwd().resolve() / "outputs"

        run_dirs = [p for p in candidate_out_root.iterdir() if p.exists() and p.is_dir()] if candidate_out_root.exists() else []
        if not run_dirs:
            raise FileNotFoundError(f"No run directories found under {candidate_out_root}")
        out_dir_path = max(run_dirs, key=lambda p: p.stat().st_mtime)

    if "REPO_ROOT" in globals():
        repo_root_path = Path(REPO_ROOT).resolve()
    else:
        notebook_root = Path.cwd().resolve()
        search_roots = [
            notebook_root,
            *notebook_root.parents,
            Path("/content/EFGP-Eigenpro"),
            Path("/content"),
        ]
        repo_root_path = next((p for p in search_roots if (p / "efgp_eigenpro_py").exists()), None)
        if repo_root_path is None:
            raise FileNotFoundError("Could not locate repo root containing 'efgp_eigenpro_py'.")

    notebook_src_path = (
        repo_root_path
        / "efgp_eigenpro_py"
        / "gpu"
        / "box_toeplitz_active_block"
        / "boxeig_inverse_diagnostics_experiment.ipynb"
    ).resolve()

    drive_mount_dir = Path(globals().get("DRIVE_MOUNT_DIR", Path("/content/drive")))
    drive_mydrive_dir = Path(globals().get("DRIVE_MYDRIVE_DIR", drive_mount_dir / "MyDrive"))
    if not drive_mydrive_dir.exists() and drive is not None:
        try:
            drive.mount(str(drive_mount_dir))
        except Exception as e:
            print("Drive mount retry skipped:", e)

    if not drive_mydrive_dir.exists():
        raise FileNotFoundError(
            "Google Drive is not mounted. Run the 'Colab / Drive setup' cell first and ensure MyDrive is available."
        )

    if "DRIVE_OUTPUT_DIR" in globals():
        drive_output_dir = Path(DRIVE_OUTPUT_DIR)
    else:
        drive_output_dir = drive_mydrive_dir / "EFGP_Eigenpro" / "benchmark_exports"
    drive_output_dir.mkdir(parents=True, exist_ok=True)

    zip_name = f"{out_dir_path.name}_bundle.zip"
    export_root = out_dir_path.parent / f"{out_dir_path.name}_export_bundle"
    zip_base = out_dir_path.parent / f"{out_dir_path.name}_bundle"
    zip_path = Path(f"{zip_base}.zip")
    target_drive_path = drive_output_dir / zip_name

    summary_csv_exists = (
        (out_dir_path / "master_summary.csv").exists()
        or (out_dir_path / "aggregate_summary.csv").exists()
        or (out_dir_path / "summary.csv").exists()
    )
    required_checks = {
        "out_dir_exists": out_dir_path.exists(),
        "notebook_exists": notebook_src_path.exists(),
        "summary_csv_exists": summary_csv_exists,
        "experiment_config_exists": (out_dir_path / "experiment_config.json").exists(),
    }
    print("artifact checks:", _json.dumps({k: bool(v) for k, v in required_checks.items()}, indent=2))
    missing = [k for k, v in required_checks.items() if not bool(v)]
    if missing:
        raise FileNotFoundError("Missing required artifacts: " + ", ".join(missing))

    if export_root.exists():
        _shutil.rmtree(export_root)
    export_root.mkdir(parents=True, exist_ok=True)

    export_results_dir = export_root / out_dir_path.name
    _shutil.copytree(out_dir_path, export_results_dir)

    notebook_dst_dir = export_root / "notebook"
    notebook_dst_dir.mkdir(parents=True, exist_ok=True)
    notebook_dst_path = notebook_dst_dir / notebook_src_path.name
    if notebook_src_path.exists():
        _shutil.copy2(notebook_src_path, notebook_dst_path)

    export_manifest = {
        "run_tag": out_dir_path.name,
        "elapsed": time_str,
        "export_root": str(export_root),
        "out_dir": str(out_dir_path),
        "notebook_src": str(notebook_src_path),
        "drive_output_dir": str(drive_output_dir),
        "github_notebook_url": globals().get("GITHUB_NOTEBOOK_URL", ""),
        "colab_notebook_url": globals().get("COLAB_NOTEBOOK_URL", ""),
        "includes": [
            str(export_results_dir),
            str(notebook_dst_path) if notebook_src_path.exists() else "notebook_missing",
        ],
    }
    (export_root / "export_manifest.json").write_text(
        _json.dumps(export_manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    if zip_path.exists():
        zip_path.unlink()
    print(f"Packing export bundle: {export_root}")
    _shutil.make_archive(str(zip_base), "zip", root_dir=str(export_root.parent), base_dir=export_root.name)
    print("zip saved:", zip_path)

    print(f"Copying zip to Google Drive: {target_drive_path}")
    try:
        _shutil.copy2(zip_path, target_drive_path)
    except Exception as e:
        print("Drive copy failed:", e)

    local_ok = zip_path.exists() and zip_path.stat().st_size > 0
    drive_ok = target_drive_path.exists() and target_drive_path.stat().st_size > 0
    print("local zip ok:", local_ok)
    print("drive zip ok:", drive_ok)

    if local_ok:
        if _HAS_COLAB_EXPORT and files is not None:
            print("Starting browser backup download...")
            files.download(str(zip_path))
        else:
            print("Not in Colab; browser download skipped.")

    print("-" * 30)
    print("Experiment status:", "completed and backed up" if drive_ok else "local zip ok, Drive sync failed")
    print("Elapsed:", time_str)
    print("OUT_DIR:", out_dir_path)
    print("Local zip:", zip_path)
    print("Drive zip:", target_drive_path)
    print("-" * 30)

    if drive_ok and _HAS_COLAB_EXPORT and runtime is not None:
        print("Disconnecting Colab runtime in 20 seconds to save GPU quota...")
        _time.sleep(20)
        runtime.unassign()
    elif not drive_ok:
        print("Drive sync check failed; automatic disconnect cancelled.")


## 10. 实验内容、参数与指标说明

### 实验方法分组

| 分组 | method | 预条件器 / 求解方式 | precompute |
|---|---|---|---|
| 默认 CG baseline | `plain_cg` | 不使用谱或 active-block 预条件器的标准 CG | original/default |
| EigenPro baseline | `eigenpro_pcg_q*` | 全局 EigenPro eigenspace preconditioner + PCG | C1 |
| inverse BTAB | `btab_auto_*` / `btab_exact_*` / `btab_inner_pcg_*` | expanded box 上 exact inverse、Cholesky solve 或 inner PCG；tail 使用 diagonal inverse | C1 |
| Box-EigenPro BTAB | `btab_boxeig_*_q*` | expanded box 上 EigenPro-type approximate inverse；tail 使用 diagonal inverse | C1 |

所有 BTAB 方法使用 `B = active.box_idx` 作为实际 active block，`R = B^c = active.tail_idx`。`active_idx` 只是由 top-k 或 tau 得到的 raw active set，随后会扩张成规则低频 box，以便使用 local Toeplitz FFT matvec。

### 本 notebook 的主要参数

- 数据集：synthetic 与 USGS 两个 family；`DATASET_STEMS` 为 family 种子，`N_TRAIN_LIST` 非空时按文件名后缀 `_ntrainN` / `_nN` 解析（与 `report_results.ipynb` 一致；restore 会尝试全部候选别名）。
- `reg_lambda`：线性系统 `A = D* G D + reg_lambda I` 中的 `sigma2`，实现中不额外乘 `N`。
- `EPS_LIST`：EFGP Fourier grid / approximation accuracy 参数。
- `tol`、`maxiter`：外层 CG / FGMRES 的停止容差与最大迭代数。
- `BTAB_CUSTOM`：`True` 启用非笛卡尔候选路线；`False` 恢复旧的 top-k × q 全笛卡尔积。
- `BTAB_CUSTOM_ROUTE`：可选 `group_a`、`group_b`、`group_c`、`schedule` 或完全手工的 `custom`。
- 手工 `custom` 使用 `BTAB_CUSTOM_INVERSE_TOPK_LIST` 与 `BTAB_CUSTOM_BOXEIG_TOPK_Q_PAIRS`，两条路线彼此独立，不再自动做笛卡尔积。
- `btab_active_mode`：选择 raw active set 的方式，当前为 `topk`；对应候选由 `btab_topk_list` 给出。
- `btab_box_budget`：expanded box 允许的最大规模约束。
- `btab_solve_mode`：inverse BTAB 的 box solve 策略；`auto` 根据 box size 选择 exact 或 inner PCG。
- `btab_eig_q_list`：Box-EigenPro 保留的 top eigenvector 数；求解器实际计算 `q+1` 个 eigenpairs。
- `btab_eig_tol`、`btab_eig_maxiter`、`btab_eig_ncv`：local `A_BB` eigensolver 的容差、最大迭代数和子空间大小。
- `btab_diagnostic_mode`：`none` 不计算 post diagnostics；`cheap` 计算低成本指标；`full` 再计算 `epsilon_T`、`eta_inv`、`eta_eig`。
- `BEST_CONFIG_METRIC`：第 8 节选择最佳 top-k 或 `(top-k, q)` 的目标字段，默认是 `time_total` 且越小越好；候选重复运行先取中位数。

### 共同的 B/R 指标

| 字段 | 定义 | 含义 |
|---|---|---|
| `btab_active_size_raw` | raw active set 的元素数 | top-k / tau 本身选出的规模，不是实际 preconditioner block size |
| `btab_box_size` | `|B|` | expanded low-frequency box 的大小，也是实际 local block 大小 |
| `tail_size` | `|R| = M - |B|` | preconditioner 中 diagonal tail block 的大小 |
| `rho_max_T` | `max(j in R) N|D_j|^2 / sigma2` | tail 中最强单个 Fourier mode；越小越接近 regularization-dominated |
| `tail_energy` | `sum(j in R) N|D_j|^2 / sigma2` | tail 的总相对能量；主要用于判断截断风险，不是 block preconditioner 的硬条件 |
| `epsilon_T` | `||Lambda_RR^(-1/2)(A_RR-Lambda_RR)Lambda_RR^(-1/2)||_2` | 检查 tail diagonal approximation 是否合理；越小越好 |

经验上 `epsilon_T < 0.1` 很好，`0.1-0.3` 通常可用；明显大于 `0.5` 表示应扩大 box 或改进 tail approximation。

### Inverse BTAB 指标

| 字段 | 定义 / 内容 | 含义 |
|---|---|---|
| `eta_inv` | `||A_BB^(-1/2) A_BR Lambda_RR^(-1/2)||_2` | exact inverse block 下的 B/R coupling；越小越好 |
| `eta_inv_sq` | `eta_inv^2` 对应 PSD operator 的最大特征值 | matrix-free power iteration 实际估计的量 |
| `solve_mode` | `exact` 或 `inner_pcg` | 当前 box block 的求解策略 |
| `exact_apply_mode` | inverse matrix 或 Cholesky solve | exact route 的 apply 实现 |
| `inner_total_iters` | 所有 inner solve 的累计迭代数 | 衡量 nested solve 的额外开销 |

`eta_inv < 0.3` 通常比较理想；如果 `epsilon_T` 小但 `eta_inv` 大，主要问题是 B/R coupling，应扩大 `B`。

### Box-EigenPro 指标

local operator 为 `A_BB v = D_B^* G_BB(D_B v) + sigma2 v`。预条件器使用 `alpha = 1/theta_(q+1)`：`P_eig v = alpha v + U_q[(Theta_q^(-1)-alpha I)U_q^*v]`。

| 字段 | 定义 / 内容 | 含义 |
|---|---|---|
| `btab_eig_q` | EigenPro top subspace 维数 `q` | 存储和 apply 成本约为 `O(q|B|)` |
| `btab_eig_theta_q1` | 第 `q+1` 大 local eigenvalue | top-q eigenvalues 被压到的目标阈值 |
| `btab_eig_theta_q1_over_sigma2` | `theta_(q+1) / sigma2` | local B block 压平后的保守条件数上界；越小越好 |
| `btab_eig_eig_residual_max` | `max_i ||A_BB u_i-theta_i u_i||/(theta_i||u_i||)` | eigensolver 的最差相对残差 |
| `btab_eig_eig_residual_median` | eigenpair 相对残差中位数 | eigenpair 整体质量 |
| `eta_eig` | `||P_eig(A_BB)^(1/2) A_BR Lambda_RR^(-1/2)||_2` | Box-EigenPro block 下的 B/R coupling；越小越好 |
| `btab_eig_n_ABB_matvec_cols` | eigensolver 传入 local block matvec 的累计列数 | `A_BB` matvec-equivalent column count，不含正交化成本 |
| `btab_eig_storage_bytes` | `U_q`、eigenvalues 和 apply 数据的存储量 | Box-EigenPro setup 的显存 / 内存指标 |

`theta_(q+1)/sigma2 < 100` 很好，`100-1000` 通常可用；更大时优先增加 `q`。若该比值已小而 `eta_eig` 仍大，则应扩大 box，而不是继续增加 `q`。

### 求解质量、时间与隔离计数

| 字段 | 含义 |
|---|---|
| `iterations` / `cg_iters` / `outer_iters` | 主求解器迭代数 |
| `n_matvec` / `cg_n_matvec` | 主流程 full operator matvec 数 |
| `n_precond` | 主流程 preconditioner apply 数 |
| `time_precompute` | NUFFT / grid precomputation 时间；CG 使用 original/default，其他方法使用 C1 |
| `time_eigenspace` / `time_eig_setup` | 全局 EigenPro或 local Box-EigenPro eigenspace setup 时间 |
| `time_precond_build` | preconditioner 构建时间，不包含 post diagnostics |
| `time_solve` | 主求解时间，不包含 post diagnostics |
| `time_predict` | prediction 时间 |
| `time_total` | 主实验总时间，不包含 post diagnostics |
| `time_post_diagnostics` | solve / predict 完成后单独运行 diagnostics 的时间 |
| `time_total_with_diagnostics` | `time_total + time_post_diagnostics` |
| `diagnostic_n_A_matvec` | diagnostics 独立 wrapper 的 full `A` matvec 数 |
| `diagnostic_n_ASS_matvec` | diagnostics 的 local `A_BB` matvec 数 |
| `diagnostic_n_precond` | diagnostics 中额外的 preconditioner / inverse apply 数 |
| `diagnostic_n_power_iter` | full spectral diagnostics 的 power-iteration 步数 |
| `rmse_train` / `rmse_test` | 训练集与测试集预测误差，用于确认加速路线没有改变解的质量 |

诊断 operator 使用独立计数器，因此不会增加主流程的 `n_matvec`、`n_precond`，也不会改变 `time_precond_build` 或 `time_solve`。